In [1]:
# Skipped in CI: Colab/bootstrap dependency install cell.


import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
# ケーススタディ: イベント同期解析

このチュートリアルでは、実務でのワークフローに即した例を紹介します。ログからのイベント抽出、時刻補正（模した例）、そしてイベントごとの解析・プロット生成までの一連の流れを `SegmentTable` で管理します。

このケーススタディでも、イベント窓は `gwpy.segments.Segment` で表し、gwexpy は `SegmentTable` による表管理と `apply()` ベースの一括処理を担当します。GWpy 基盤クラスと gwexpy 拡張の役割分担は [SegmentTable: 基本](intro_segment_table.ipynb) と同じです。


In [2]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import warnings

with warnings.catch_warnings():
    warnings.simplefilter('ignore')

    import pandas as pd
    from gwpy.segments import Segment

    from gwexpy.table import SegmentTable

    # 1. Simulate finding events
    events_df = pd.DataFrame({
    "gps": [1234567890.1, 1234567895.5, 1234567900.2],
    "snr": [15, 8, 22]
    })

    # 2. Create SegmentTable from events
    # Define a 4-second window around each GPS time
    segs = [Segment(t-2, t+2) for t in events_df["gps"]]
    st = SegmentTable.from_segments(segs, snr=events_df["snr"])
    st


## バッチプロット生成

`apply()` を使って、各イベントのズームプロットなどを一括生成し、そのファイルパスをテーブル自体に保存できます。

In [3]:
import os

os.makedirs("outputs", exist_ok=True)

def generate_event_plot(row):
    # Simulate plotting
    path = f"outputs/event_{row.index}.png"
    # plot = row["raw"].plot() -> plot.save(path)
    with open(path, "w") as f: f.write("Dummy PNG")
    return {"plot_path": path}

st_plots = st.apply(generate_event_plot)
st_plots.display()

,span,snr,plot_path
0,"(1234567888.1, 1234567892.1)",15,outputs/event_0.png
1,"(1234567893.5, 1234567897.5)",8,outputs/event_1.png
2,"(1234567898.2, 1234567902.2)",22,outputs/event_2.png


## 結果の絞り込みと出力

最後に、特定の条件（SNR > 10 など）でテーブルをフィルタリングし、結果をエクスポートします。

In [4]:
st_best = st_plots.select(mask=st_plots.to_pandas()["snr"] > 10)
st_best.to_pandas()

,span,snr,plot_path
0,"(1234567888.1, 1234567892.1)",15,outputs/event_0.png
1,"(1234567898.2, 1234567902.2)",22,outputs/event_2.png
